In [3]:
pip install -U langchain-text-splitters

In [4]:
pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.5/124.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.0/567.0 kB 11.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.5.5
    Uninstalling langchain-core-1.5.5:
      Successfully uninstalled langchain-core-1.5.5


In [3]:
pip install -U tiktoken

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import numpy as np
import pandas as pd
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
import nltk
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
import math

In [4]:
def buscar_arquivos():
    pasta = Path(r"/content/")
    caminhos = list(pasta.glob("*.md"))
    return caminhos

In [5]:
caminhos = [str(arquivo) for arquivo in buscar_arquivos()]
caminhos

['/content/gpt3_language_models.md',
 '/content/retrieval_augmented_generation.md',
 '/content/attention_is_all_you_need.md',
 '/content/lora_low_rank_adaptation.md',
 '/content/bert_pretraining.md',
 '/content/scaling_laws_llm.md',
 '/content/escrita_academica_ia.md',
 '/content/gpt4_technical_report.md',
 '/content/twitter_algoritmo.md',
 '/content/llama_foundation_models.md',
 '/content/instruct_gpt.md',
 '/content/bioetica_e_ia.md']

In [6]:
def contar_tokens_dos_blocos(lista_de_blocos, modelo="text-embedding-3-small"):
    codificador = tiktoken.encoding_for_model(modelo)

    total_tokens = 0
    tokens_por_bloco = []

    for texto_puro in lista_de_blocos:
        num_tokens = len(codificador.encode(texto_puro, disallowed_special=()))
        tokens_por_bloco.append(num_tokens)
        total_tokens += num_tokens

    media_tokens = total_tokens / len(lista_de_blocos) if lista_de_blocos else 0

    return total_tokens

In [7]:
def gerar_embeddings(termos):

  load_dotenv()
  client = OpenAI(
      api_key=''
      )

  embeddings = []

  response = client.embeddings.create(
      model="text-embedding-3-small", input=[str(t) for t in termos]
  )

  embeddings = [item.embedding for item in response.data]

  return np.array(embeddings)

In [8]:
def markdown(caminho, chunk_size_max=24000, overlap=0):

  with open(caminho, 'r', encoding='utf-8') as f:
    conteudo_do_texto = f.read()

  headers_para_dividir = [
      ("#", "Header 1"),
      ("##", "Header 2"),
      ("###", "Header 3"),
  ]

  markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_para_dividir)
  documentos_markdown = markdown_splitter.split_text(conteudo_do_texto)

  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size_max,
      chunk_overlap=overlap,
      separators=["\n\n", "\n", " ", ""]
  )
  documentos_finais = text_splitter.split_documents(documentos_markdown)

  dados = []
  for doc in documentos_finais:
      dados.append(doc.page_content)

  return dados

In [9]:
parametros = ['paragrafo_recursivo(caminho)', 'sentencas_agrupadas(caminho)', 'markdown(caminho)']

In [13]:
def gerar_base_dados_metadados(lista_caminhos):
    resultados_dados_metadados = {}

    for caminho in lista_caminhos:
        chave = caminho

        p_recursivo = paragrafo_recursivo(caminho)
        s_agrupadas = sentencas_agrupadas(caminho)
        md = markdown(caminho)

        resultados_dados_metadados[chave] = {
            "paragrafo_recursivo": p_recursivo,
            "sentencas_agrupadas": s_agrupadas,
            "markdown": md
        }

    return resultados_dados_metadados

resultados = gerar_base_dados_metadados(caminhos)

In [ ]:
pr = resultados['/content/bioetica_e_ia.md']['paragrafo_recursivo']
sa = resultados['/content/bioetica_e_ia.md']['sentencas_agrupadas']
md = resultados['/content/bioetica_e_ia.md']['markdown']

In [14]:
def infos(dados):

  # Total de chunks
  print("Total de chunks: " + str(len(dados)))

  # Tamanho medio chunck
  tamanho_medio = sum(len(s) for s in dados) / len(dados)
  print("tamanho médio: " + str(tamanho_medio))

  # Menor chunks
  menor_documento = min(dados, key=len)
  print("menor string: " + str(len(menor_documento)))

  # Maior chunks
  maior_documento = max(dados, key=len)
  print("maior string: " + str(len(maior_documento)))

  # Numeros de tokens
  print("total de tokens: " + str(contar_tokens_dos_blocos(dados)))

In [ ]:
infos(pr)

Total de chunks: 73
tamanho médio: 20379.698630136987
menor string: 1004
maior string: 40447
total de tokens: 347328


In [ ]:
infos(sa)

Total de chunks: 135
tamanho médio: 377.4888888888889
menor string: 52
maior string: 1088
total de tokens: 14761


In [ ]:
infos(md)

Total de chunks: 22
tamanho médio: 2278.1363636363635
menor string: 20
maior string: 8579
total de tokens: 14513


In [ ]:
import os
import json

def criar_json(chuncks, embeddings, nome_parametros):
  dados_json = [
      {
          "Texto": texto,
          "Embeddings": emb.tolist() if hasattr(emb, "tolist") else emb,
      }
      for texto, emb in zip(chuncks, embeddings)
  ]

  pasta_destino = "resultados_json"

  os.makedirs(pasta_destino, exist_ok=True)

  nome = os.path.join(pasta_destino, f"{nome_parametros}.json")

  with open(nome, 'w', encoding='utf-8') as f:
      json.dump(dados_json, f, indent=4, ensure_ascii=False, default=str)

  print(f"JSON gerado com sucesso em: {nome}")

# Embeddings sentença agrupada manual

In [11]:
def sentencas_agrupadas(caminho, chunk_size_max=20000):
  dados_arquivo = []

  nltk.download('punkt', quiet=True)
  nltk.download('punkt_tab', quiet=True)

  with open(caminho, 'r', encoding='utf-8') as f:
    conteudo_do_texto = f.read()

  todas_sentencas = nltk.sent_tokenize(conteudo_do_texto, language='portuguese')

  tamanho_grupo = 3
  documentos_finais = []
  contador_bloco = 1

  for i in range(0, len(todas_sentencas), tamanho_grupo):
      grupo = todas_sentencas[i : i + tamanho_grupo]
      texto_bloco = " ".join(grupo)

      if len(texto_bloco) > chunk_size_max:
          texto_bloco = texto_bloco[:chunk_size_max]

      doc = Document(
          page_content=texto_bloco,
          metadata={
              "chunk_index": contador_bloco,
              "total_sentences": len(grupo),
              "character_count": len(texto_bloco)
          }
      )
      documentos_finais.append(doc)
      contador_bloco += 1

  for doc in documentos_finais:
      dados_arquivo.append(doc.page_content)

  return dados_arquivo

In [ ]:
print("sentencas_agrupadas" + str(caminhos[0]))
dados_sentencas_agrupadas = []
dados_sentencas_agrupadas = sentencas_agrupadas(caminhos[0])
embeddings_sentencas_agrupadas = gerar_embeddings(dados_sentencas_agrupadas)
nome = str('sentencas_agrupadas_'+str(os.path.basename(caminhos[0])))
criar_json(dados_sentencas_agrupadas, embeddings_sentencas_agrupadas, nome)
print("-------------------------------------------------------------")
print("\n")

sentencas_agrupadas/content/bioetica_e_ia.md
JSON gerado com sucesso em: resultados_json/sentencas_agrupadas_bioetica_e_ia.md.json
-------------------------------------------------------------




In [ ]:
print("sentencas_agrupadas" + str(caminhos[1]))
dados_sentencas_agrupadas = []
dados_sentencas_agrupadas = sentencas_agrupadas(caminhos[1])
embeddings_sentencas_agrupadas = gerar_embeddings(dados_sentencas_agrupadas)
nome = str('sentencas_agrupadas_'+str(os.path.basename(caminhos[1])))
criar_json(dados_sentencas_agrupadas, embeddings_sentencas_agrupadas, nome)
print("-------------------------------------------------------------")
print("\n")

sentencas_agrupadas/content/twitter_algoritmo.md
JSON gerado com sucesso em: resultados_json/sentencas_agrupadas_twitter_algoritmo.md.json
-------------------------------------------------------------




In [ ]:
print("sentencas_agrupadas" + str(caminhos[2]))
dados_sentencas_agrupadas = []
dados_sentencas_agrupadas = sentencas_agrupadas(caminhos[2])
embeddings_sentencas_agrupadas = gerar_embeddings(dados_sentencas_agrupadas)
nome = str('sentencas_agrupadas_'+str(os.path.basename(caminhos[2])))
criar_json(dados_sentencas_agrupadas, embeddings_sentencas_agrupadas, nome)
print("-------------------------------------------------------------")
print("\n")

sentencas_agrupadas/content/instruct_gpt.md
JSON gerado com sucesso em: resultados_json/sentencas_agrupadas_instruct_gpt.md.json
-------------------------------------------------------------




In [ ]:
print("sentencas_agrupadas" + str(caminhos[3]))
dados_sentencas_agrupadas = []
dados_sentencas_agrupadas = sentencas_agrupadas(caminhos[3])
embeddings_sentencas_agrupadas = gerar_embeddings(dados_sentencas_agrupadas)
nome = str('sentencas_agrupadas_'+str(os.path.basename(caminhos[3])))
criar_json(dados_sentencas_agrupadas, embeddings_sentencas_agrupadas, nome)
print("-------------------------------------------------------------")
print("\n")

sentencas_agrupadas/content/retrieval_augmented_generation.md
JSON gerado com sucesso em: resultados_json/sentencas_agrupadas_retrieval_augmented_generation.md.json
-------------------------------------------------------------




In [ ]:
print("sentencas_agrupadas" + str(caminhos[4]))
dados_sentencas_agrupadas = []
dados_sentencas_agrupadas = sentencas_agrupadas(caminhos[4])
embeddings_sentencas_agrupadas = gerar_embeddings(dados_sentencas_agrupadas)
nome = str('sentencas_agrupadas_'+str(os.path.basename(caminhos[4])))
criar_json(dados_sentencas_agrupadas, embeddings_sentencas_agrupadas, nome)
print("-------------------------------------------------------------")
print("\n")

sentencas_agrupadas/content/bert_pretraining.md
JSON gerado com sucesso em: resultados_json/sentencas_agrupadas_bert_pretraining.md.json
-------------------------------------------------------------




In [ ]:
print("sentencas_agrupadas" + str(caminhos[5]))
dados_sentencas_agrupadas = []
dados_sentencas_agrupadas = sentencas_agrupadas(caminhos[5])
embeddings_sentencas_agrupadas = gerar_embeddings(dados_sentencas_agrupadas)
nome = str('sentencas_agrupadas_'+str(os.path.basename(caminhos[5])))
criar_json(dados_sentencas_agrupadas, embeddings_sentencas_agrupadas, nome)
print("-------------------------------------------------------------")
print("\n")

sentencas_agrupadas/content/scaling_laws_llm.md
JSON gerado com sucesso em: resultados_json/sentencas_agrupadas_scaling_laws_llm.md.json
-------------------------------------------------------------




In [ ]:
print("sentencas_agrupadas" + str(caminhos[6]))
dados_sentencas_agrupadas = []
dados_sentencas_agrupadas = sentencas_agrupadas(caminhos[6])
embeddings_sentencas_agrupadas = gerar_embeddings(dados_sentencas_agrupadas)
nome = str('sentencas_agrupadas_'+str(os.path.basename(caminhos[6])))
criar_json(dados_sentencas_agrupadas, embeddings_sentencas_agrupadas, nome)
print("-------------------------------------------------------------")
print("\n")

sentencas_agrupadas/content/attention_is_all_you_need.md
JSON gerado com sucesso em: resultados_json/sentencas_agrupadas_attention_is_all_you_need.md.json
-------------------------------------------------------------




In [ ]:
print("sentencas_agrupadas" + str(caminhos[7]))
dados_sentencas_agrupadas = []
dados_sentencas_agrupadas = sentencas_agrupadas(caminhos[7])
embeddings_sentencas_agrupadas = gerar_embeddings(dados_sentencas_agrupadas)
nome = str('sentencas_agrupadas_'+str(os.path.basename(caminhos[7])))
criar_json(dados_sentencas_agrupadas, embeddings_sentencas_agrupadas, nome)
print("-------------------------------------------------------------")
print("\n")

sentencas_agrupadas/content/llama_foundation_models.md
JSON gerado com sucesso em: resultados_json/sentencas_agrupadas_llama_foundation_models.md.json
-------------------------------------------------------------




In [ ]:
print("sentencas_agrupadas" + str(caminhos[8]))
dados_sentencas_agrupadas = []
dados_sentencas_agrupadas = sentencas_agrupadas(caminhos[8])
embeddings_sentencas_agrupadas = gerar_embeddings(dados_sentencas_agrupadas)
nome = str('sentencas_agrupadas_'+str(os.path.basename(caminhos[8])))
criar_json(dados_sentencas_agrupadas, embeddings_sentencas_agrupadas, nome)
print("-------------------------------------------------------------")
print("\n")

sentencas_agrupadas/content/escrita_academica_ia.md
JSON gerado com sucesso em: resultados_json/sentencas_agrupadas_escrita_academica_ia.md.json
-------------------------------------------------------------




In [ ]:
print("sentencas_agrupadas" + str(caminhos[9]))
dados_sentencas_agrupadas = []
dados_sentencas_agrupadas = sentencas_agrupadas(caminhos[9])
embeddings_sentencas_agrupadas = gerar_embeddings(dados_sentencas_agrupadas)
nome = str('sentencas_agrupadas_'+str(os.path.basename(caminhos[9])))
criar_json(dados_sentencas_agrupadas, embeddings_sentencas_agrupadas, nome)
print("-------------------------------------------------------------")
print("\n")

sentencas_agrupadas/content/lora_low_rank_adaptation.md
JSON gerado com sucesso em: resultados_json/sentencas_agrupadas_lora_low_rank_adaptation.md.json
-------------------------------------------------------------




In [ ]:
print("sentencas_agrupadas" + str(caminhos[10]))
dados_sentencas_agrupadas = []
dados_sentencas_agrupadas = sentencas_agrupadas(caminhos[10])
embeddings_sentencas_agrupadas = gerar_embeddings(dados_sentencas_agrupadas)
nome = str('sentencas_agrupadas_'+str(os.path.basename(caminhos[10])))
criar_json(dados_sentencas_agrupadas, embeddings_sentencas_agrupadas, nome)
print("-------------------------------------------------------------")
print("\n")

sentencas_agrupadas/content/gpt4_technical_report.md
JSON gerado com sucesso em: resultados_json/sentencas_agrupadas_gpt4_technical_report.md.json
-------------------------------------------------------------




In [ ]:
print("sentencas_agrupadas" + str(caminhos[11]))
dados_sentencas_agrupadas = []
dados_sentencas_agrupadas = sentencas_agrupadas(caminhos[11])
embeddings_sentencas_agrupadas = gerar_embeddings(dados_sentencas_agrupadas)
nome = str('sentencas_agrupadas_'+str(os.path.basename(caminhos[11])))
criar_json(dados_sentencas_agrupadas, embeddings_sentencas_agrupadas, nome)
print("-------------------------------------------------------------")
print("\n")

sentencas_agrupadas/content/bert_pretraining.md
JSON gerado com sucesso em: resultados_json/sentencas_agrupadas_bert_pretraining.md.json
-------------------------------------------------------------




# Embeddings markdown manual




In [ ]:
caminho = caminhos[0]
print("markdown" + str(caminho))
dados_markdown = []
dados_markdown = markdown(caminho)
embeddings_markdown = gerar_embeddings(dados_markdown)
nome = str('markdown_'+str(os.path.basename(caminho)))
criar_json(dados_markdown, embeddings_markdown, nome)
print("-------------------------------------------------------------")
print("\n")

In [ ]:
caminho = caminhos[1]
print("markdown" + str(caminho))
dados_markdown = []
dados_markdown = markdown(caminho)
embeddings_markdown = gerar_embeddings(dados_markdown)
nome = str('markdown_'+str(os.path.basename(caminho)))
criar_json(dados_markdown, embeddings_markdown, nome)
print("-------------------------------------------------------------")
print("\n")

In [ ]:
caminho = caminhos[2]
print("markdown" + str(caminho))
dados_markdown = []
dados_markdown = markdown(caminho)
embeddings_markdown = gerar_embeddings(dados_markdown)
nome = str('markdown_'+str(os.path.basename(caminho)))
criar_json(dados_markdown, embeddings_markdown, nome)
print("-------------------------------------------------------------")
print("\n")

In [ ]:
caminho = caminhos[3]
print("markdown" + str(caminho))
dados_markdown = []
dados_markdown = markdown(caminho)
embeddings_markdown = gerar_embeddings(dados_markdown)
nome = str('markdown_'+str(os.path.basename(caminho)))
criar_json(dados_markdown, embeddings_markdown, nome)
print("-------------------------------------------------------------")
print("\n")

In [ ]:
caminho = caminhos[4]
print("markdown" + str(caminho))
dados_markdown = []
dados_markdown = markdown(caminho)
embeddings_markdown = gerar_embeddings(dados_markdown)
nome = str('markdown_'+str(os.path.basename(caminho)))
criar_json(dados_markdown, embeddings_markdown, nome)
print("-------------------------------------------------------------")
print("\n")

In [ ]:
caminho = caminhos[5]
print("markdown" + str(caminho))
dados_markdown = []
dados_markdown = markdown(caminho)
embeddings_markdown = gerar_embeddings(dados_markdown)
nome = str('markdown_'+str(os.path.basename(caminho)))
criar_json(dados_markdown, embeddings_markdown, nome)
print("-------------------------------------------------------------")
print("\n")

In [ ]:
caminho = caminhos[6]
print("markdown" + str(caminho))
dados_markdown = []
dados_markdown = markdown(caminho)
embeddings_markdown = gerar_embeddings(dados_markdown)
nome = str('markdown_'+str(os.path.basename(caminho)))
criar_json(dados_markdown, embeddings_markdown, nome)
print("-------------------------------------------------------------")
print("\n")

In [ ]:
caminho = caminhos[7]
print("markdown" + str(caminho))
dados_markdown = []
dados_markdown = markdown(caminho)
embeddings_markdown = gerar_embeddings(dados_markdown)
nome = str('markdown_'+str(os.path.basename(caminho)))
criar_json(dados_markdown, embeddings_markdown, nome)
print("-------------------------------------------------------------")
print("\n")

In [ ]:
caminho = caminhos[8]
print("markdown" + str(caminho))
dados_markdown = []
dados_markdown = markdown(caminho)
embeddings_markdown = gerar_embeddings(dados_markdown)
nome = str('markdown_'+str(os.path.basename(caminho)))
criar_json(dados_markdown, embeddings_markdown, nome)
print("-------------------------------------------------------------")
print("\n")

In [ ]:
caminho = caminhos[9]
print("markdown" + str(caminho))
dados_markdown = []
dados_markdown = markdown(caminho)
embeddings_markdown = gerar_embeddings(dados_markdown)
nome = str('markdown_'+str(os.path.basename(caminho)))
criar_json(dados_markdown, embeddings_markdown, nome)
print("-------------------------------------------------------------")
print("\n")

In [ ]:
caminho = caminhos[10]
print("markdown" + str(caminho))
dados_markdown = []
dados_markdown = markdown(caminho)
embeddings_markdown = gerar_embeddings(dados_markdown)
nome = str('markdown_'+str(os.path.basename(caminho)))
criar_json(dados_markdown, embeddings_markdown, nome)
print("-------------------------------------------------------------")
print("\n")

In [ ]:
caminho = caminhos[11]
print("markdown" + str(caminho))
dados_markdown = []
dados_markdown = markdown(caminho)
embeddings_markdown = gerar_embeddings(dados_markdown)
nome = str('markdown_'+str(os.path.basename(caminho)))
criar_json(dados_markdown, embeddings_markdown, nome)
print("-------------------------------------------------------------")
print("\n")

# Embeddings paragrafo_recursivo

In [12]:
def paragrafo_recursivo(caminho):
  dados = []
  divisor = RecursiveCharacterTextSplitter(
      chunk_size=20000,
      chunk_overlap=0,
      separators=["\n\n", "\n", " ", ""]
  )
  with open(caminho, 'r', encoding='utf-8') as f:
    conteudo_do_texto = f.read()

  chunks = divisor.split_text(conteudo_do_texto)
  dados.extend(chunks)
  return dados

In [ ]:
caminho = caminhos[0]
print("paragrafo_recursivo" + str(caminho))
dados_paragrafo_recursivo = []
dados_paragrafo_recursivo = paragrafo_recursivo(caminho)
embeddings_paragrafo_recursivo = gerar_embeddings(dados_paragrafo_recursivo)
nome = str('paragrafo_recursivo'+str(os.path.basename(caminho)))
criar_json(dados_paragrafo_recursivo, embeddings_paragrafo_recursivo, nome)
print("-------------------------------------------------------------")
print("\n")

paragrafo_recursivo/content/gpt3_language_models.md
JSON gerado com sucesso em: resultados_json/paragrafo_recursivogpt3_language_models.md.json
-------------------------------------------------------------




In [ ]:
caminho = caminhos[1]
print("paragrafo_recursivo" + str(caminho))
dados_paragrafo_recursivo = []
dados_paragrafo_recursivo = paragrafo_recursivo(caminho)
embeddings_paragrafo_recursivo = gerar_embeddings(dados_paragrafo_recursivo)
nome = str('paragrafo_recursivo'+str(os.path.basename(caminho)))
criar_json(dados_paragrafo_recursivo, embeddings_paragrafo_recursivo, nome)
print("-------------------------------------------------------------")
print("\n")

paragrafo_recursivo/content/retrieval_augmented_generation.md
JSON gerado com sucesso em: resultados_json/paragrafo_recursivoretrieval_augmented_generation.md.json
-------------------------------------------------------------




In [ ]:
caminho = caminhos[2]
print("paragrafo_recursivo" + str(caminho))
dados_paragrafo_recursivo = []
dados_paragrafo_recursivo = paragrafo_recursivo(caminho)
embeddings_paragrafo_recursivo = gerar_embeddings(dados_paragrafo_recursivo)
nome = str('paragrafo_recursivo'+str(os.path.basename(caminho)))
criar_json(dados_paragrafo_recursivo, embeddings_paragrafo_recursivo, nome)
print("-------------------------------------------------------------")
print("\n")

paragrafo_recursivo/content/attention_is_all_you_need.md
JSON gerado com sucesso em: resultados_json/paragrafo_recursivoattention_is_all_you_need.md.json
-------------------------------------------------------------




In [ ]:
caminho = caminhos[3]
print("paragrafo_recursivo" + str(caminho))
dados_paragrafo_recursivo = []
dados_paragrafo_recursivo = paragrafo_recursivo(caminho)
embeddings_paragrafo_recursivo = gerar_embeddings(dados_paragrafo_recursivo)
nome = str('paragrafo_recursivo'+str(os.path.basename(caminho)))
criar_json(dados_paragrafo_recursivo, embeddings_paragrafo_recursivo, nome)
print("-------------------------------------------------------------")
print("\n")

paragrafo_recursivo/content/lora_low_rank_adaptation.md
JSON gerado com sucesso em: resultados_json/paragrafo_recursivolora_low_rank_adaptation.md.json
-------------------------------------------------------------




In [ ]:
caminho = caminhos[4]
print("paragrafo_recursivo" + str(caminho))
dados_paragrafo_recursivo = []
dados_paragrafo_recursivo = paragrafo_recursivo(caminho)
embeddings_paragrafo_recursivo = gerar_embeddings(dados_paragrafo_recursivo)
nome = str('paragrafo_recursivo'+str(os.path.basename(caminho)))
criar_json(dados_paragrafo_recursivo, embeddings_paragrafo_recursivo, nome)
print("-------------------------------------------------------------")
print("\n")

paragrafo_recursivo/content/bert_pretraining.md
JSON gerado com sucesso em: resultados_json/paragrafo_recursivobert_pretraining.md.json
-------------------------------------------------------------




In [ ]:
caminho = caminhos[5]
print("paragrafo_recursivo" + str(caminho))
dados_paragrafo_recursivo = []
dados_paragrafo_recursivo = paragrafo_recursivo(caminho)
embeddings_paragrafo_recursivo = gerar_embeddings(dados_paragrafo_recursivo)
nome = str('paragrafo_recursivo'+str(os.path.basename(caminho)))
criar_json(dados_paragrafo_recursivo, embeddings_paragrafo_recursivo, nome)
print("-------------------------------------------------------------")
print("\n")

paragrafo_recursivo/content/scaling_laws_llm.md
JSON gerado com sucesso em: resultados_json/paragrafo_recursivoscaling_laws_llm.md.json
-------------------------------------------------------------




In [ ]:
caminho = caminhos[6]
print("paragrafo_recursivo" + str(caminho))
dados_paragrafo_recursivo = []
dados_paragrafo_recursivo = paragrafo_recursivo(caminho)
embeddings_paragrafo_recursivo = gerar_embeddings(dados_paragrafo_recursivo)
nome = str('paragrafo_recursivo'+str(os.path.basename(caminho)))
criar_json(dados_paragrafo_recursivo, embeddings_paragrafo_recursivo, nome)
print("-------------------------------------------------------------")
print("\n")

paragrafo_recursivo/content/escrita_academica_ia.md
JSON gerado com sucesso em: resultados_json/paragrafo_recursivoescrita_academica_ia.md.json
-------------------------------------------------------------




In [ ]:
caminho = caminhos[7]
print("paragrafo_recursivo" + str(caminho))
dados_paragrafo_recursivo = []
dados_paragrafo_recursivo = paragrafo_recursivo(caminho)
embeddings_paragrafo_recursivo = gerar_embeddings(dados_paragrafo_recursivo)
nome = str('paragrafo_recursivo'+str(os.path.basename(caminho)))
criar_json(dados_paragrafo_recursivo, embeddings_paragrafo_recursivo, nome)
print("-------------------------------------------------------------")
print("\n")

paragrafo_recursivo/content/gpt4_technical_report.md
JSON gerado com sucesso em: resultados_json/paragrafo_recursivogpt4_technical_report.md.json
-------------------------------------------------------------




In [ ]:
caminho = caminhos[8]
print("paragrafo_recursivo" + str(caminho))
dados_paragrafo_recursivo = []
dados_paragrafo_recursivo = paragrafo_recursivo(caminho)
embeddings_paragrafo_recursivo = gerar_embeddings(dados_paragrafo_recursivo)
nome = str('paragrafo_recursivo'+str(os.path.basename(caminho)))
criar_json(dados_paragrafo_recursivo, embeddings_paragrafo_recursivo, nome)
print("-------------------------------------------------------------")
print("\n")

paragrafo_recursivo/content/twitter_algoritmo.md
JSON gerado com sucesso em: resultados_json/paragrafo_recursivotwitter_algoritmo.md.json
-------------------------------------------------------------




In [ ]:
caminho = caminhos[9]
print("paragrafo_recursivo" + str(caminho))
dados_paragrafo_recursivo = []
dados_paragrafo_recursivo = paragrafo_recursivo(caminho)
embeddings_paragrafo_recursivo = gerar_embeddings(dados_paragrafo_recursivo)
nome = str('paragrafo_recursivo'+str(os.path.basename(caminho)))
criar_json(dados_paragrafo_recursivo, embeddings_paragrafo_recursivo, nome)
print("-------------------------------------------------------------")
print("\n")

paragrafo_recursivo/content/llama_foundation_models.md
JSON gerado com sucesso em: resultados_json/paragrafo_recursivollama_foundation_models.md.json
-------------------------------------------------------------




In [ ]:
caminho = caminhos[10]
print("paragrafo_recursivo" + str(caminho))
dados_paragrafo_recursivo = []
dados_paragrafo_recursivo = paragrafo_recursivo(caminho)
embeddings_paragrafo_recursivo = gerar_embeddings(dados_paragrafo_recursivo)
nome = str('paragrafo_recursivo'+str(os.path.basename(caminho)))
criar_json(dados_paragrafo_recursivo, embeddings_paragrafo_recursivo, nome)
print("-------------------------------------------------------------")
print("\n")

paragrafo_recursivo/content/instruct_gpt.md
JSON gerado com sucesso em: resultados_json/paragrafo_recursivoinstruct_gpt.md.json
-------------------------------------------------------------




In [ ]:
caminho = caminhos[11]
print("paragrafo_recursivo" + str(caminho))
dados_paragrafo_recursivo = []
dados_paragrafo_recursivo = paragrafo_recursivo(caminho)
embeddings_paragrafo_recursivo = gerar_embeddings(dados_paragrafo_recursivo)
nome = str('paragrafo_recursivo'+str(os.path.basename(caminho)))
criar_json(dados_paragrafo_recursivo, embeddings_paragrafo_recursivo, nome)
print("-------------------------------------------------------------")
print("\n")

paragrafo_recursivo/content/bioetica_e_ia.md
JSON gerado com sucesso em: resultados_json/paragrafo_recursivobioetica_e_ia.md.json
-------------------------------------------------------------




# Infos

In [15]:
def infos(dados):

  # Total de chunks
  print("Total de chunks: " + str(len(dados)))

  # Tamanho medio chunck
  tamanho_medio = sum(len(s) for s in dados) / len(dados)
  print("tamanho médio: " + str(tamanho_medio))

  # Menor chunks
  menor_documento = min(dados, key=len)
  print("menor string: " + str(len(menor_documento)))

  # Maior chunks
  maior_documento = max(dados, key=len)
  print("maior string: " + str(len(maior_documento)))

  # Numeros de tokens
  print("total de tokens: " + str(contar_tokens_dos_blocos(dados)))

In [16]:
caminhos

['/content/gpt3_language_models.md',
 '/content/retrieval_augmented_generation.md',
 '/content/attention_is_all_you_need.md',
 '/content/lora_low_rank_adaptation.md',
 '/content/bert_pretraining.md',
 '/content/scaling_laws_llm.md',
 '/content/escrita_academica_ia.md',
 '/content/gpt4_technical_report.md',
 '/content/twitter_algoritmo.md',
 '/content/llama_foundation_models.md',
 '/content/instruct_gpt.md',
 '/content/bioetica_e_ia.md']

In [17]:
caminhos = [str(arquivo) for arquivo in buscar_arquivos()]

for caminho in caminhos:
    print(f"==================================================")
    print(f" ATIVIDADE PARA O ARQUIVO: {caminho}")
    print(f"==================================================")
    print("markdown")
    print(infos(markdown(caminho)))
    print("sentencas_agrupadas")
    print(infos(sentencas_agrupadas(caminho)))
    print("paragrafo_recursivo")
    print(infos(paragrafo_recursivo(caminho)))



 ATIVIDADE PARA O ARQUIVO: /content/gpt3_language_models.md
markdown
Total de chunks: 62
tamanho médio: 5361.854838709677
menor string: 262
maior string: 23877
total de tokens: 69078
None
sentencas_agrupadas
Total de chunks: 639
tamanho médio: 502.69796557120503
menor string: 5
maior string: 20000
total de tokens: 67710
None
paragrafo_recursivo
Total de chunks: 20
tamanho médio: 16689.35
menor string: 1925
maior string: 19902
total de tokens: 69221
None
 ATIVIDADE PARA O ARQUIVO: /content/retrieval_augmented_generation.md
markdown
Total de chunks: 33
tamanho médio: 2150.5151515151515
menor string: 100
maior string: 23229
total de tokens: 17451
None
sentencas_agrupadas
Total de chunks: 219
tamanho médio: 326.73059360730593
menor string: 44
maior string: 1753
total de tokens: 17668
None
paragrafo_recursivo
Total de chunks: 4
tamanho médio: 17988.5
menor string: 14542
maior string: 19832
total de tokens: 17668
None
 ATIVIDADE PARA O ARQUIVO: /content/attention_is_all_you_need.md
markdown
